<a href="https://colab.research.google.com/github/BakhturinaPolina/billionaire_romance_study/blob/main/Goodreads_Datasets_Romance_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
"""
Updated merge and clean pipeline for Goodreads romance dataset.

This script streams edition metadata, groups by work_id, merges editions
using rules for ratings and shelves, applies language and date filters,
and produces a consolidated CSV per work.  Diagnostics CSVs and a
provenance JSON are also generated for validation.  Improvements include:

  * Use `np.asarray` in `weighted_median` to avoid NumPy 2.0 copy warnings.
  * Always compute an unweighted mean of edition ratings when possible.
  * Normalize language codes to lowercase.
  * Cast inferred publication year and median page counts to integers.
  * Silence fastText/Numpy errors in `detect_lang_fasttext` by returning
    ("unknown", 0.0) on any exception.
  * At the end, save a trimmed metadata CSV with only essential columns.

Before running on the full dataset, ensure Drive is mounted and adjust
BOOKS_PATH/REVIEWS_PATH as needed.  Set RUN_SUBSET=False to process
the entire corpus.
"""

# ---------- Mount Drive ----------
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ---------- Installs ----------
!pip install -q fasttext==0.9.2 rapidfuzz matplotlib seaborn

# ---------- Imports ----------
import os
import json
import gzip
import csv
import datetime
import logging
import re
import warnings
from collections import defaultdict, Counter
from tqdm import tqdm
import pandas as pd
import numpy as np
from rapidfuzz import fuzz
import fasttext
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# ---------- Configuration (edit paths / params) ----------
BOOKS_PATH = "/content/drive/MyDrive/Goodreads_Metadata_Reviews_2017/goodreads_books_romance.json.gz"
REVIEWS_PATH = "/content/drive/MyDrive/Goodreads_Metadata_Reviews_2017/goodreads_reviews_romance.json.gz"
OUTPUT_DIR = "/content/drive/MyDrive/Goodreads_Metadata_Reviews_2017/processing_outputs_merged_v4"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Process only a small subset of works unless RUN_SUBSET is set to False
RUN_SUBSET = True
SUBSET_WORK_COUNT = 10

TOP_SHELVES_N = 10
LANG_PROB_THRESHOLD = 0.85
MIN_YEAR = 2000

EMMA_AUTHOR_CLAUSE = "austen"
EMMA_AUTHOR_IDS_KNOWN = {"1265"}

EXCLUDE_COLUMNS = {
    "isbn", "isbn13", "asin", "kindle_asin",
    "url", "link", "image_url",
    "edition_information", "format", "country_code", "publisher"
}

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
LOGFILE = os.path.join(OUTPUT_DIR, f"merge_clean_v4_{ts}.log")
OUT_MERGED_CSV = os.path.join(OUTPUT_DIR, f"goodreads_merged_by_work_v4_{ts}.csv")
OUT_PROVENANCE_JSON = os.path.join(OUTPUT_DIR, f"work_to_editions_provenance_v4_{ts}.json")
OUT_EXCLUDED_WORKS = os.path.join(OUTPUT_DIR, f"excluded_works_v4_{ts}.csv")
OUT_NEEDS_ENRICH = os.path.join(OUTPUT_DIR, f"needs_year_enrichment_v4_{ts}.csv")
OUT_DIAG_CSV = os.path.join(OUTPUT_DIR, f"diag_problem_works_v4_{ts}.csv")
OUT_REV_COUNTS = os.path.join(OUTPUT_DIR, f"filtered_review_counts_per_work_v4_{ts}.csv")
FIG_DIR = os.path.join(OUTPUT_DIR, "figs")
os.makedirs(FIG_DIR, exist_ok=True)

# ---------- Logging ----------
logger = logging.getLogger("merge_clean_v4")
logger.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.INFO)
ch.setFormatter(logging.Formatter('[%(levelname)s] %(message)s'))
logger.handlers = [ch]
fh = logging.FileHandler(LOGFILE, encoding='utf-8')
fh.setLevel(logging.DEBUG)
fh.setFormatter(logging.Formatter('%(asctime)s [%(levelname)s] %(message)s'))
logger.addHandler(fh)

logger.info("Starting merge+clean pipeline v4 with diagnostics and weighted-median support (updated).")

# ---------- Helper functions ----------
def iter_json_gz(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                logger.debug("Skipping malformed JSON line.")
                continue

def to_int_safe(x):
    if x in (None, "", []):
        return None
    try:
        return int(str(x).replace(",", ""))
    except Exception:
        try:
            return int(float(x))
        except Exception:
            return None

def to_float_safe(x):
    try:
        return float(x)
    except Exception:
        return None

def normalize_text_tag(s):
    if not s:
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"[_\-/]", " ", s)
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def weighted_median(values, weights):
    if not values:
        return None
    vals = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    if w.sum() == 0:
        return None
    order = np.argsort(vals)
    vals_sorted = vals[order]
    w_sorted = w[order]
    cumw = np.cumsum(w_sorted)
    cutoff = w_sorted.sum() / 2.0
    idx = np.searchsorted(cumw, cutoff)
    idx = min(idx, len(vals_sorted) - 1)
    return float(vals_sorted[idx])

def merge_popular_shelves_normalized(all_shelves_lists, fuzzy_threshold=90):
    raw_to_count = defaultdict(int)
    raw_examples = defaultdict(set)
    for lst in all_shelves_lists:
        if not isinstance(lst, list):
            continue
        for itm in lst:
            try:
                if isinstance(itm, dict):
                    raw = str(itm.get("name", "")).strip()
                    cnt = to_int_safe(itm.get("count")) or 0
                else:
                    raw = str(itm).strip()
                    cnt = 1
                if not raw:
                    continue
                raw_to_count[raw] += cnt
                raw_examples[raw].add(raw)
            except Exception:
                continue
    norm_map = {raw: normalize_text_tag(raw) for raw in raw_to_count}
    norm_counts = defaultdict(int)
    norm_examples = defaultdict(set)
    for raw, cnt in raw_to_count.items():
        norm = norm_map.get(raw, "")
        if norm:
            norm_counts[norm] += cnt
            norm_examples[norm].add(raw)
    norms = list(norm_counts.keys())
    rep = {}
    for n in norms:
        if n in rep:
            continue
        rep[n] = n
        for m in norms:
            if m == n or m in rep:
                continue
            score = fuzz.token_sort_ratio(n, m)
            if score >= fuzzy_threshold:
                rep[m] = n
                norm_counts[n] += norm_counts.get(m, 0)
                norm_examples[n].update(norm_examples.get(m, set()))
    reps = set(rep.values())
    final_counts = {r: norm_counts[r] for r in reps}
    final_examples = {r: list(norm_examples.get(r, [])) for r in reps}
    sorted_list = sorted(final_counts.items(), key=lambda x: x[1], reverse=True)
    sorted_dict = {k: v for k, v in sorted_list}
    return sorted_dict, final_examples

# Load fastText model (download if necessary)
FT_MODEL_PATH = "/content/lid.176.bin"
if not os.path.exists(FT_MODEL_PATH):
    logger.info("Downloading fastText lid.176.bin (~90MB)...")
    !wget -q -O /content/lid.176.bin https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
    logger.info("Download complete.")

warnings.filterwarnings(
    "ignore",
    message="Unable to avoid copy while creating an array as requested",
    category=UserWarning,
)

ft_model = None
try:
    ft_model = fasttext.load_model(FT_MODEL_PATH)
    logger.info("fastText model loaded.")
except Exception as e:
    logger.exception("fastText model load failed: %s" % e)
    ft_model = None

def detect_lang_fasttext(text):
    if not ft_model or not text or text.strip() == "":
        return ("unknown", 0.0)
    try:
        labels, probs = ft_model.predict(text.replace("\n", " "), k=1)
        if isinstance(labels, (list, tuple)):
            lang = labels[0].replace("__label__", "")
            prob = float(probs[0])
        else:
            lang = str(labels).replace("__label__", "")
            prob = float(probs)
        return (lang, prob)
    except Exception:
        return ("unknown", 0.0)

# ---------- Phase 1: streaming editions and grouping by work_id ----------
logger.info("Phase 1: streaming editions and grouping by work_id (this may take a while)...")
work_map = defaultdict(list)
lines = 0
for obj in tqdm(iter_json_gz(BOOKS_PATH), desc="reading editions"):
    lines += 1
    book_id = str(obj.get('book_id')) if obj.get('book_id') is not None else ""
    work_id = str(obj.get('work_id')) if obj.get('work_id') is not None else ""
    obj['_book_id'] = book_id
    obj['_work_id'] = work_id
    obj['_ratings_count_int'] = to_int_safe(obj.get('ratings_count'))
    obj['_text_reviews_count_int'] = to_int_safe(obj.get('text_reviews_count'))
    obj['_publication_year_int'] = to_int_safe(obj.get('publication_year'))
    obj['_average_rating_float'] = to_float_safe(obj.get('average_rating'))
    obj['_num_pages_int'] = to_int_safe(obj.get('num_pages'))
    work_map[work_id].append(obj)
logger.info(f"Streamed {lines} edition rows. Found {len(work_map)} distinct work_ids.")

# ---------- Subset selection ----------
work_ids_ordered = [w for w in work_map.keys() if w]
if RUN_SUBSET:
    selected_work_ids = work_ids_ordered[:SUBSET_WORK_COUNT]
    logger.info(f"RUN_SUBSET=True -> processing first {len(selected_work_ids)} works.")
else:
    selected_work_ids = work_ids_ordered
    logger.info(f"Processing all {len(selected_work_ids)} works.")

# Pre-clean stats
pre_n_works = len(selected_work_ids)
pre_n_editions = sum(len(work_map[w]) for w in selected_work_ids)
pre_authors = set()
for w in selected_work_ids:
    for e in work_map[w]:
        a = e.get('authors')
        if a:
            if isinstance(a, list):
                for ai in a:
                    if isinstance(ai, dict):
                        if ai.get('author_id'):
                            pre_authors.add(str(ai.get('author_id')))
                        elif ai.get('name'):
                            pre_authors.add(ai.get('name').strip().lower())
                    else:
                        pre_authors.add(str(ai).strip().lower())
            elif isinstance(a, dict):
                if a.get('author_id'):
                    pre_authors.add(str(a.get('author_id')))
                elif a.get('name'):
                    pre_authors.add(a.get('name').strip().lower())
            else:
                pre_authors.add(str(a).strip().lower())
logger.info(f"Pre-clean summary: works={pre_n_works}, editions={pre_n_editions}, distinct_authors_est={len(pre_authors)}")

# ---------- Title clustering probe ----------
logger.info("Running title clustering probe (normalized title -> work_ids) on selected subset...")
title_to_works = defaultdict(list)
for w in selected_work_ids:
    some_title = None
    for e in work_map[w]:
        if e.get('title'):
            some_title = e.get('title')
            break
    if not some_title:
        continue
    nt = normalize_text_tag(some_title)
    title_to_works[nt].append(w)

clusters = {nt: ids for nt, ids in title_to_works.items() if len(ids) > 1}
if clusters:
    logger.info(f"[TITLE-CLUSTER] Found {len(clusters)} normalized titles mapping to >1 work_id (sample):")
    for nt, ids in list(clusters.items())[:10]:
        logger.info(f"  '{nt}' -> {ids}")
else:
    logger.info("No title clusters (same normalized title -> multiple work_ids) found in subset.")

# ---------- Prepare diagnostic CSV writer ----------
diag_csv_fh = open(OUT_DIAG_CSV, "w", encoding='utf-8', newline="")
diag_writer = csv.writer(diag_csv_fh)
diag_writer.writerow([
    "work_id", "num_editions", "publication_year_inferred",
    "ratings_sum", "ratings_max", "text_reviews_sum", "text_reviews_max",
    "distinct_title_count", "frac_editions_zero_reviews", "any_edition_en",
    "needs_year_enrichment", "authors_ids_or_names", "exclude_reasons"
])

# ---------- Phase 2: merge per work with rules & diagnostics ----------
logger.info("Phase 2: merging editions per work with preservation rules and diagnostics...")
merged_rows = []
provenance = {}
excluded_work_reasons = {}

for wid in tqdm(selected_work_ids, desc="merging works"):
    editions = work_map.get(wid, [])
    if not editions:
        excluded_work_reasons[wid] = ["no_editions"]
        continue
    provenance[wid] = [
        {
            "book_id": e.get('_book_id'),
            "publication_year": e.get('publication_year'),
            "ratings_count": e.get('ratings_count'),
            "text_reviews_count": e.get('text_reviews_count'),
            "average_rating": e.get('average_rating'),
            "language_code": e.get('language_code'),
        }
        for e in editions
    ]
    edition_book_ids = [e.get('_book_id') for e in editions if e.get('_book_id')]
    num_editions = len(editions)
    pub_years = [e['_publication_year_int'] for e in editions if e.get('_publication_year_int') is not None]
    publication_year_inferred = int(min(pub_years)) if pub_years else None
    ratings_sum = sum([e.get('_ratings_count_int') or 0 for e in editions])
    ratings_max = max([e.get('_ratings_count_int') for e in editions if e.get('_ratings_count_int') is not None], default=None)
    reviews_sum = sum([e.get('_text_reviews_count_int') or 0 for e in editions])
    reviews_max = max([e.get('_text_reviews_count_int') for e in editions if e.get('_text_reviews_count_int') is not None], default=None)
    avg_vals = []
    avg_wts = []
    for e in editions:
        avg = e.get('_average_rating_float')
        wt = e.get('_ratings_count_int') or 0
        if avg is not None:
            avg_vals.append(avg)
            avg_wts.append(wt)
    average_rating_weighted_mean = None
    average_rating_weighted_median = None
    average_rating_unweighted_mean = None
    if avg_vals:
        total_w = sum(avg_wts)
        if total_w > 0:
            average_rating_weighted_mean = sum(v * w for v, w in zip(avg_vals, avg_wts)) / total_w
        average_rating_weighted_median = weighted_median(avg_vals, avg_wts)
        average_rating_unweighted_mean = sum(avg_vals) / len(avg_vals)
    titles = [e.get('title') for e in editions if e.get('title')]
    title = None
    if titles:
        tcounts = Counter(titles).most_common()
        top_count = tcounts[0][1]
        top_titles = [t for t, c in tcounts if c == top_count]
        if len(top_titles) == 1:
            title = top_titles[0]
        else:
            best = None
            best_rc = -1
            for e in editions:
                if e.get('title') in top_titles:
                    rc = e.get('_ratings_count_int') or 0
                    if rc > best_rc:
                        best_rc = rc
                        best = e.get('title')
            title = best or top_titles[0]
    title_without_series = None
    tws = [e.get('title_without_series') for e in editions if e.get('title_without_series')]
    if tws:
        title_without_series = Counter(tws).most_common(1)[0][0]
    descs = [e.get('description') for e in editions if e.get('description') and str(e.get('description')).strip()]
    description = max(descs, key=lambda s: len(str(s))) if descs else None
    desc_source_book_ids = [e.get('_book_id') for e in editions if e.get('description') == description] if description else []
    pages = [e.get('_num_pages_int') for e in editions if e.get('_num_pages_int') is not None]
    num_pages_med = int(np.median(pages)) if pages else None
    language_codes = sorted(set([str(e.get('language_code')).strip().lower() for e in editions if e.get('language_code')]))
    any_edition_en = any((lc or "").lower().startswith("en") for lc in language_codes)
    series_set = set()
    for e in editions:
        s = e.get('series')
        if s:
            if isinstance(s, list):
                for si in s:
                    series_set.add(str(si))
            else:
                series_set.add(str(s))
    series_union = sorted(series_set)
    similar_books_set = set()
    for e in editions:
        sb = e.get('similar_books')
        if sb:
            if isinstance(sb, list):
                similar_books_set.update([str(x) for x in sb])
            else:
                similar_books_set.add(str(sb))
    similar_books_union = sorted(similar_books_set)
    shelves_counts, shelf_examples = merge_popular_shelves_normalized([e.get('popular_shelves') for e in editions])
    top_shelves = [name for name, cnt in sorted(shelves_counts.items(), key=lambda x: x[1], reverse=True)][:TOP_SHELVES_N]
    author_union = set()
    for e in editions:
        a_field = e.get('authors')
        if a_field:
            if isinstance(a_field, list):
                for a in a_field:
                    if isinstance(a, dict):
                        aid = a.get('author_id') or a.get('id')
                        name = a.get('name') or a.get('author_name')
                        if aid:
                            author_union.add(str(aid))
                        elif name:
                            author_union.add(str(name).strip().lower())
                    else:
                        author_union.add(str(a).strip().lower())
            elif isinstance(a_field, dict):
                aid = a_field.get('author_id') or a_field.get('id')
                name = a_field.get('name') or a_field.get('author_name')
                if aid:
                    author_union.add(str(aid))
                elif name:
                    author_union.add(str(name).strip().lower())
            else:
                author_union.add(str(a_field))
    authors_ids_or_names = "|".join(sorted(author_union)) if author_union else None
    canonical_book_id = None
    best_rating = -1
    for e in editions:
        rc = e.get('_ratings_count_int') or 0
        if rc > best_rating:
            best_rating = rc
            canonical_book_id = e.get('_book_id')
    canonical = next((e for e in editions if e.get('_book_id') == canonical_book_id), editions[0])
    all_keys = set()
    for e in editions:
        all_keys.update([k for k in e.keys() if not k.startswith('_')])
    final_row = {}
    for k in sorted(all_keys):
        if k in EXCLUDE_COLUMNS:
            continue
        if k in ('authors', 'popular_shelves', 'series', 'similar_books'):
            continue
        if canonical.get(k) not in (None, ""):
            if k == 'language_code':
                final_row[k] = str(canonical.get(k)).strip().lower()
            else:
                final_row[k] = canonical.get(k)
        else:
            vals = [e.get(k) for e in editions if e.get(k) not in (None, "", [], {})]
            final_row[k] = Counter(vals).most_common(1)[0][0] if vals else None
    final_row['work_id'] = wid
    final_row['edition_book_ids'] = "|".join(edition_book_ids)
    final_row['num_editions_for_work'] = num_editions
    final_row['publication_year_inferred'] = int(publication_year_inferred) if publication_year_inferred is not None else None
    final_row['publication_year_chosen'] = final_row['publication_year_inferred']
    final_row['ratings_count_sum'] = int(ratings_sum)
    final_row['ratings_count_max'] = int(ratings_max) if ratings_max is not None else None
    final_row['text_reviews_count_sum'] = int(reviews_sum)
    final_row['text_reviews_count_max'] = int(reviews_max) if reviews_max is not None else None
    final_row['average_rating_weighted_mean'] = round(average_rating_weighted_mean, 4) if average_rating_weighted_mean is not None else None
    final_row['average_rating_weighted_median'] = round(average_rating_weighted_median, 4) if average_rating_weighted_median is not None else None
    final_row['average_rating_unweighted_mean'] = round(average_rating_unweighted_mean, 4) if average_rating_unweighted_mean is not None else None
    final_row['title'] = title if title else final_row.get('title')
    final_row['title_without_series'] = title_without_series if title_without_series else final_row.get('title_without_series')
    final_row['description'] = description if description else final_row.get('description')
    final_row['description_source_book_ids'] = "|".join(desc_source_book_ids) if desc_source_book_ids else None
    final_row['description_length'] = len(final_row['description']) if final_row.get('description') else 0
    final_row['num_pages_median'] = int(num_pages_med) if num_pages_med is not None else None
    final_row['language_codes'] = "|".join(language_codes) if language_codes else None
    final_row['series_union'] = "|".join(series_union) if series_union else None
    final_row['similar_books_union'] = "|".join(similar_books_union) if similar_books_union else None
    final_row['all_shelves_agg'] = json.dumps(shelves_counts, ensure_ascii=False)
    final_row['top_shelves'] = "|".join(top_shelves)
    final_row['authors_ids_or_names'] = authors_ids_or_names
    final_row['canonical_book_id'] = canonical_book_id
    final_row['_editions_provenance'] = provenance[wid]
    exclude_reasons = []
    norm_title = normalize_text_tag(final_row.get('title') or "")
    is_emma = False
    if norm_title == "emma":
        author_tokens = "|".join([p.lower() for p in (authors_ids_or_names or "").split("|")])
        if any(aid in author_tokens for aid in EMMA_AUTHOR_IDS_KNOWN) or EMMA_AUTHOR_CLAUSE in author_tokens:
            is_emma = True
        else:
            shelf_keys_concat = " ".join(shelves_counts.keys()).lower()
            if EMMA_AUTHOR_CLAUSE in shelf_keys_concat:
                is_emma = True
    if is_emma:
        exclude_reasons.append("Emma_by_Jane_Austen")
    lang_ok = False
    if any_edition_en:
        lang_ok = True
    else:
        sample_text = (" ".join(filter(None, [final_row.get('title', ''), final_row.get('description', '')])) or "")[:4000]
        lang, prob = detect_lang_fasttext(sample_text)
        if lang == "en" and prob >= LANG_PROB_THRESHOLD:
            lang_ok = True
    if not lang_ok:
        exclude_reasons.append("non_english")
        logger.info(f"[LANG-EXCLUDE] work_id={wid} languages_seen={list(set(language_codes))} fasttext_sample_prob={prob if 'prob' in locals() else 'NA'}")
    needs_enrichment = False
    if publication_year_inferred is None:
        needs_enrichment = True
    else:
        if publication_year_inferred < MIN_YEAR:
            exclude_reasons.append("year_before_2000")
            logger.info(f"[YEAR-EXCLUDE] work_id={wid} inferred_year={publication_year_inferred}")
    if exclude_reasons:
        excluded_work_reasons[wid] = exclude_reasons
        zero_reviews = sum(1 for e in editions if (e.get('_text_reviews_count_int') or 0) == 0)
        frac_zero = zero_reviews / max(1, len(editions))
        distinct_titles = sorted(set([str(e.get('title') or "").strip() for e in editions if e.get('title')]))
        diag_writer.writerow([
            wid, len(editions), publication_year_inferred or "",
            ratings_sum, ratings_max or "", reviews_sum, reviews_max or "",
            len(distinct_titles), round(frac_zero, 3), any_edition_en,
            needs_enrichment, authors_ids_or_names or "", ";".join(exclude_reasons),
        ])
        continue
    final_row['needs_year_enrichment'] = needs_enrichment
    zero_reviews = sum(1 for e in editions if (e.get('_text_reviews_count_int') or 0) == 0)
    frac_zero = zero_reviews / max(1, len(editions))
    distinct_titles = sorted(set([str(e.get('title') or "").strip() for e in editions if e.get('title')]))
    diag_writer.writerow([
        wid, len(editions), publication_year_inferred or "",
        ratings_sum, ratings_max or "", reviews_sum, reviews_max or "",
        len(distinct_titles), round(frac_zero, 3), any_edition_en,
        needs_enrichment, authors_ids_or_names or "", "",
    ])
    merged_rows.append(final_row)

# Close diag CSV
diag_csv_fh.close()
logger.info(f"Phase 2 complete. Kept works: {len(merged_rows)}; Excluded: {len(excluded_work_reasons)}")
logger.info(f"Diagnostic CSV saved to: {OUT_DIAG_CSV}")

# ---------- Save provenance & exclusion lists ----------
logger.info("Saving provenance and exclusion lists...")
with open(OUT_PROVENANCE_JSON, "w", encoding='utf-8') as fh:
    json.dump(provenance, fh, ensure_ascii=False, indent=2)
with open(OUT_EXCLUDED_WORKS, "w", newline='', encoding='utf-8') as cf:
    writer = csv.writer(cf)
    writer.writerow(["work_id", "reasons"])
    for wid, reasons in excluded_work_reasons.items():
        writer.writerow([wid, ";".join(reasons)])
with open(OUT_NEEDS_ENRICH, "w", newline='', encoding='utf-8') as nef:
    w = csv.writer(nef)
    w.writerow(["work_id", "note"])
    for row in merged_rows:
        if row.get('needs_year_enrichment'):
            w.writerow([row.get('work_id'), "missing_publication_year_inferred"])
logger.info(f"Saved provenance JSON to {OUT_PROVENANCE_JSON}")
logger.info(f"Saved excluded works to {OUT_EXCLUDED_WORKS}")
logger.info(f"Saved needs_year_enrichment to {OUT_NEEDS_ENRICH}")

# ---------- Save merged CSV and trimmed final CSV ----------
logger.info("Saving merged CSV (dropping _editions_provenance for file size)...")
if merged_rows:
    for r in merged_rows:
        if '_editions_provenance' in r:
            del r['_editions_provenance']
    merged_df = pd.DataFrame(merged_rows)
    merged_df.to_csv(OUT_MERGED_CSV, index=False, encoding='utf-8')
    logger.info(f"Saved merged CSV to {OUT_MERGED_CSV}")

    # Define columns to keep for the final metadata CSV
    cols_to_keep = [
        'work_id',
        'title',
        'title_without_series',
        'authors_ids_or_names',
        'publication_year_inferred',
        'num_editions_for_work',
        'num_pages_median',
        'language_codes',
        'ratings_count_sum',
        'text_reviews_count_sum',
        'average_rating_weighted_mean',
        'average_rating_weighted_median',
        'description',
        'description_length',
        'top_shelves',
        'needs_year_enrichment',
        'canonical_book_id',
    ]
    # Construct path and save trimmed DataFrame
    OUT_FINAL_CSV = os.path.join(OUTPUT_DIR, f"goodreads_metadata_final_v4_{ts}.csv")
    df_final = merged_df[[c for c in cols_to_keep if c in merged_df.columns]]
    df_final.to_csv(OUT_FINAL_CSV, index=False, encoding='utf-8')
    logger.info(f"Saved final metadata CSV (trimmed) to {OUT_FINAL_CSV}")
else:
    merged_df = pd.DataFrame()
    logger.info("No merged rows to save (empty result).")

# ---------- Phase 3: count English reviews ----------
logger.info("Phase 3: streaming reviews and counting English reviews per kept work (fastText). This may take time for full dataset.")
kept_work_ids = set([r['work_id'] for r in merged_rows])
book_to_work = {}
for wid in kept_work_ids:
    for e in provenance.get(wid, []):
        bid = e.get('book_id')
        if bid:
            book_to_work[str(bid)] = wid
review_counts_per_work = Counter()
if os.path.exists(REVIEWS_PATH):
    for rev in tqdm(iter_json_gz(REVIEWS_PATH), desc="streaming reviews"):
        bid = str(rev.get('book_id') or rev.get('bookId') or "")
        wid = book_to_work.get(bid)
        if not wid:
            continue
        text = rev.get('review_text') or rev.get('body') or rev.get('review') or ""
        if not text:
            continue
        lang, prob = detect_lang_fasttext(text[:2000])
        if lang == "en" and prob >= LANG_PROB_THRESHOLD:
            review_counts_per_work[wid] += 1
else:
    logger.warning(f"Reviews file not found at {REVIEWS_PATH}; skipping review language filtering and counts.")
with open(OUT_REV_COUNTS, "w", encoding='utf-8') as fh:
    fh.write("work_id,filtered_english_review_count\n")
    for wid, cnt in review_counts_per_work.items():
        fh.write(f"{wid},{cnt}\n")
logger.info(f"Saved filtered review counts to {OUT_REV_COUNTS}")

# ---------- Phase 4: statistics & visualizations ----------
logger.info("Phase 4: computing summary statistics & visualizations...")
post_n_works = len(merged_rows)
post_n_editions = sum([len(provenance[w]) for w in [r['work_id'] for r in merged_rows]])
post_authors = set()
for r in merged_rows:
    a = r.get('authors_ids_or_names') or ""
    for p in a.split("|"):
        if p:
            post_authors.add(p)
post_n_authors = len(post_authors)

pre_ratings_sum = sum([sum([to_int_safe(e.get('ratings_count')) or 0 for e in work_map[w]]) for w in selected_work_ids])
pre_reviews_sum = sum([sum([to_int_safe(e.get('text_reviews_count')) or 0 for e in work_map[w]]) for w in selected_work_ids])
post_ratings_sum = sum([r.get('ratings_count_sum') or 0 for r in merged_rows])
post_reviews_sum = sum([r.get('text_reviews_count_sum') or 0 for r in merged_rows])

QR_PATH = os.path.join(OUTPUT_DIR, f"merge_clean_v4_quality_{ts}.txt")
with open(QR_PATH, "w", encoding='utf-8') as fh:
    fh.write("\n".join([
        f"Timestamp: {ts}",
        f"Selected works processed: {len(selected_work_ids)}",
        f"Pre-clean: works={pre_n_works}, editions={pre_n_editions}, distinct_authors_est={len(pre_authors)}",
        f"Post-clean: works={post_n_works}, editions_covered_by_post={post_n_editions}, distinct_authors_post={post_n_authors}",
        f"Ratings sum: before={pre_ratings_sum}, after={post_ratings_sum}",
        f"Text reviews sum: before={pre_reviews_sum}, after={post_reviews_sum}",
        f"Excluded works count={len(excluded_work_reasons)}",
    ]))
logger.info(f"Saved quality report to {QR_PATH}")

# Visualizations
plt.figure(figsize=(6, 4))
sns.barplot(x=["before", "after"], y=[pre_n_works, post_n_works])
plt.title("Works before vs after cleaning")
plt.ylabel("Number of works")
plt.savefig(os.path.join(FIG_DIR, f"works_before_after_{ts}.png"))
plt.close()
plt.figure(figsize=(6, 4))
sns.barplot(x=["before", "after"], y=[len(pre_authors), post_n_authors])
plt.title("Authors before vs after cleaning")
plt.savefig(os.path.join(FIG_DIR, f"authors_before_after_{ts}.png"))
plt.close()
plt.figure(figsize=(6, 4))
vals = [pre_ratings_sum if pre_ratings_sum > 0 else 1, post_ratings_sum if post_ratings_sum > 0 else 1]
sns.barplot(x=["before", "after"], y=vals)
plt.yscale("log")
plt.title("Ratings sum (log scale)")
plt.savefig(os.path.join(FIG_DIR, f"ratings_sum_before_after_{ts}.png"))
plt.close()
if merged_rows:
    years = [r.get('publication_year_inferred') for r in merged_rows if r.get('publication_year_inferred')]
    if years:
        plt.figure(figsize=(8, 4))
        sns.histplot(years, bins=30)
        plt.title("Publication year (inferred) - post-clean")
        plt.savefig(os.path.join(FIG_DIR, f"pub_year_post_{ts}.png"))
        plt.close()
logger.info(f"Saved visualizations to {FIG_DIR}")

# ---------- Finish ----------
logger.info("=== RUN SUMMARY ===")
logger.info(f"Input file: {BOOKS_PATH}")
logger.info(f"Output merged CSV: {OUT_MERGED_CSV}")
logger.info(f"Provenance JSON: {OUT_PROVENANCE_JSON}")
logger.info(f"Excluded works CSV: {OUT_EXCLUDED_WORKS}")
logger.info(f"Needs year enrichment CSV: {OUT_NEEDS_ENRICH}")
logger.info(f"Diagnostic CSV: {OUT_DIAG_CSV}")
logger.info(f"Filtered reviews counts: {OUT_REV_COUNTS}")
logger.info(f"Quality report: {QR_PATH}")
logger.info(f"Final metadata CSV: {OUT_FINAL_CSV}")
logger.info("REMINDER: commit this script and small outputs to your git repo before running full corpus (set RUN_SUBSET=False).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[INFO] Starting merge+clean pipeline v4 with diagnostics and weighted-median support (updated).
INFO:merge_clean_v4:Starting merge+clean pipeline v4 with diagnostics and weighted-median support (updated).
[INFO] fastText model loaded.
INFO:merge_clean_v4:fastText model loaded.
[INFO] Phase 1: streaming editions and grouping by work_id (this may take a while)...
INFO:merge_clean_v4:Phase 1: streaming editions and grouping by work_id (this may take a while)...
reading editions: 335449it [00:35, 9519.34it/s] 
[INFO] Streamed 335449 edition rows. Found 189420 distinct work_ids.
INFO:merge_clean_v4:Streamed 335449 edition rows. Found 189420 distinct work_ids.
[INFO] RUN_SUBSET=True -> processing first 10 works.
INFO:merge_clean_v4:RUN_SUBSET=True -> processing first 10 works.
[INFO] Pre-clean summary: works=10, editions=265, distinct_authors_est=107
INFO:merge_clean_v4:Pre-clean summary: works=10, editions=265, distinct_authors_est=107
[INFO] Running title clustering probe (normalized title